# set 模型探針：humanv0 主幹向量抽取

`aicup/setmodel_trunk.py` 對每局目標玩家第 31～180 手等距取 30 手，每手一個 768 維向量（`act_trunkfinal` 輸出的平均加最大值）。

- 這份 notebook 是實驗，不在競賽一最終版本流程內。
- 外部資源：KataGo `python/`（tag `v1.15.0`，MIT）與 `b18c384nbt-humanv0.ckpt`。checkpoint 先找 Drive 快取，沒有才下載
- 輸出在 Drive 的 `humansl-analysis/setmodel/`，每 200 局一個 `part-XXXXXX.npz`，中斷後重跑同一個 `MODE` 會接續
- 速度由 CPU 核心數決定（編碼是純 Python），GPU 大多在等

| MODE | 內容 |
|---|---|
| `bench` | 只量速度，可略過 |
| `eval` | dev 玩家（val648 加 newdev）的局，輸出 `trunk-eval/` |
| `train` | train 切分的玩家，每級 `PLAYERS_PER_RANK` 位、跳過前 `SKIP_PER_RANK` 位，輸出 `trunk-train-<名稱>/` |

## 需要的檔案

路徑都在 Drive 的 `MyDrive/aicup-2026/` 下。

| 路徑 | MODE | 產生方式 |
|---|---|---|
| `training-v1.1/training/train_*.csv`（10 個） | 全部 | 官方訓練集 v1.1，見 README；`training/` 沒有 `HD[n]`，不能用 |
| `humansl-analysis/val-v180000-dev-p3.jsonl` | `eval` | 本機 `cat humansl-analysis/val648-p3.jsonl humansl-analysis/hist6d-newdev/newdev-p3-hist.jsonl > humansl-analysis/val-v180000-dev-p3.jsonl`，兩個來源見 `colab_humansl.ipynb` 的 `val648`、`newdev_hist` |

## 參數

In [ ]:
REPO_URL = 'https://github.com/Jx-study/aicup-2026-go.git'
COMMIT = 'origin/main'  # 重現特定版本時填 commit hash
MODE = 'eval'  # 'bench'、'eval' 或 'train'
PLAYERS_PER_RANK = 0  # train 每級人數，0 表示每級全取
SKIP_PER_RANK = 350  # 去掉每級名單前 N 位，0 表示不跳過
AUTO_UNASSIGN = True

if MODE not in ('bench', 'eval', 'train'):
    raise SystemExit('MODE 要是 bench、eval 或 train')
TRAIN_DIR = '/content/training-v1.1/training'
OUT = 'humansl-analysis/setmodel'
KEYS = 'humansl-analysis/val-v180000-dev-p3.jsonl'
NEEDS = [('training-v1.1/training', '官方訓練集 v1.1，見 README')]
BASE = ['--train-dir', TRAIN_DIR, '--checkpoint', '/content/models/b18c384nbt-humanv0.ckpt']
if MODE == 'bench':
    CMD = ['python', '-m', 'aicup.setmodel_trunk', 'bench', *BASE, '--set', 'train', '--positions', '1000']
elif MODE == 'eval':
    NEEDS.append((KEYS, 'cat humansl-analysis/val648-p3.jsonl humansl-analysis/hist6d-newdev/newdev-p3-hist.jsonl > ' + KEYS))
    CMD = ['python', '-m', 'aicup.setmodel_trunk', 'extract', *BASE, '--set', 'eval', '--keys-from', KEYS,
           '--out', OUT + '/trunk-eval']
else:
    name = 'new' if SKIP_PER_RANK else (PLAYERS_PER_RANK or 'all')
    CMD = ['python', '-m', 'aicup.setmodel_trunk', 'extract', *BASE, '--set', 'train', '--players-per-rank',
           str(PLAYERS_PER_RANK), '--skip-per-rank', str(SKIP_PER_RANK), '--out', '%s/trunk-train-%s' % (OUT, name)]
print(' '.join(CMD))

## 取得程式

要看到：Drive 掛載完成、最新的 commit、GPU 型號。

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
REPO = '/content/aicup-2026-go'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO], check=True)
os.chdir(REPO)
subprocess.run(['git', 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', 'checkout', '-q', COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sgfmill'], check=True)
from aicup.colab_nb import DRIVE_ROOT, check_commit, link_drive, need, run

GIT_HEAD = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()

!git log --oneline -1
!nvidia-smi -L

## 準備資料

要看到：`humansl-analysis` 的連結、需要的檔案都在、10 個訓練 CSV 複製到 `/content`、CPU 核心數、humanv0 的路徑。

In [ ]:
import shutil

link_drive(['humansl-analysis'])
need([(os.path.join(DRIVE_ROOT, rel), how) for rel, how in NEEDS])
shutil.copytree(os.path.join(DRIVE_ROOT, 'training-v1.1/training'), TRAIN_DIR, dirs_exist_ok=True)
n_csv = len([f for f in os.listdir(TRAIN_DIR) if f.startswith('train_') and f.endswith('.csv')])
assert n_csv == 10, '%s 有 %d 個 train_*.csv' % (TRAIN_DIR, n_csv)
os.makedirs(OUT, exist_ok=True)
!nproc
KATAGO = '/content/KataGo'
KATAGO_COMMIT = 'c560d38585b446bc93b0a62c77364bc8e7d266a4'  # tag v1.15.0
if not os.path.exists(KATAGO + '/python/features.py'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', 'v1.15.0', '--filter=blob:none', '--sparse',
                    'https://github.com/lightvector/KataGo', KATAGO], check=True)
    subprocess.run(['git', '-C', KATAGO, 'sparse-checkout', 'set', 'python'], check=True)
head = subprocess.run(['git', '-C', KATAGO, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert head == KATAGO_COMMIT, 'KataGo 的 commit 不是 v1.15.0'
os.environ['KATAGO_PYTHON'] = KATAGO + '/python'

# katagotraining.org 會擋 urllib，先用 Drive 快取，沒有才用 wget
HUMANV0 = '/content/models/b18c384nbt-humanv0.ckpt'
CACHE = DRIVE_ROOT + '/katago/b18c384nbt-humanv0.ckpt'
os.makedirs(os.path.dirname(HUMANV0), exist_ok=True)
if not os.path.exists(HUMANV0):
    if os.path.exists(CACHE):
        shutil.copy(CACHE, HUMANV0)
    else:
        !wget -q -O {HUMANV0} https://media.katagotraining.org/uploaded/networks/models_extra/b18c384nbt-humanv0.ckpt
from aicup import katago_torch as kt

kt.ensure_humanv0(HUMANV0)  # 檢查大小與 SHA-256
if not os.path.exists(CACHE):
    os.makedirs(os.path.dirname(CACHE), exist_ok=True)
    shutil.copy(HUMANV0, CACHE)
print('humanv0', HUMANV0)

## 執行

輸出同時寫到 Drive 的 `humansl-analysis/setmodel/<MODE>.log`。

In [ ]:
run(CMD, os.path.join(OUT, MODE + '.log'))

## 檢查

要看到：每個輸出目錄的 part 數與最近一次的 `timing.json`。

In [ ]:
import glob, json

for d in sorted(glob.glob(OUT + '/trunk-*')):
    parts = glob.glob(d + '/part-*.npz')
    t = {}
    if os.path.exists(d + '/timing.json'):
        with open(d + '/timing.json') as f:
            t = json.load(f)
    print(d, '%d 個 part' % len(parts), t.get('runs', [])[-1:])

## 釋放 runtime

跑完才執行。程式結束後 runtime 不會自動中斷，閒置也會扣運算單元。

In [ ]:
from google.colab import runtime

drive.flush_and_unmount()  # 等 Drive 寫入完成
if AUTO_UNASSIGN:
    runtime.unassign()
else:
    print('未釋放 runtime，用完記得手動中斷')